In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("glenmore_traffic.csv")
df.info()

In [ ]:
# convert the study date to datetime for plotting
df["STUDY_DATE"] = pd.to_datetime(df["STUDY_DATE"], format=r"%m/%d/%Y %I:%M:%S %p")

In [ ]:
for dir in ["E", "W"]:
    this_dir = df[df["DIRECTION"] == dir]
    plt.plot(this_dir["STUDY_DATE"], this_dir["VOLUME"], label=dir)

plt.legend()
# zoom in so we can see a bit more detail
plt.xlim(pd.to_datetime(["2023-01-01", "2023-02-1"]))

In [ ]:
# pull the date/month/time info out of the datetime object
def extract_date_features(df):
    df["DAY_OF_WEEK"] = df["STUDY_DATE"].dt.weekday  # coded 0 to 6
    df["DAY_OF_MONTH"] = df["STUDY_DATE"].dt.day
    df["MONTH"] = df["STUDY_DATE"].dt.month
    df["TIME_HOUR"] = df["STUDY_DATE"].dt.hour
    df["TIME_MINUTES"] = df["STUDY_DATE"].dt.minute

    return df.drop(columns="STUDY_DATE")


In [ ]:
# Maybe the day of the week is important?
date_df = extract_date_features(df)
date_df.groupby(["DAY_OF_WEEK", "DIRECTION"])["VOLUME"].sum()

In [ ]:
# try a super basic model
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

train, val = train_test_split(df, test_size=0.2)

model = RandomForestRegressor()

pipeline = make_pipeline (
        FunctionTransformer(extract_date_features),
        make_column_transformer(
            (OneHotEncoder(sparse_output=False, drop="if_binary"), ["DIRECTION"]),
            ("passthrough", ["DAY_OF_MONTH", "MONTH", "TIME_HOUR"])
        ),
        model,
).set_output(transform="pandas")

pipeline.fit(train, train["VOLUME"])

In [ ]:
pipeline[:-1].transform(train)

In [ ]:
plt.scatter(train["VOLUME"], pipeline.predict(train))

In [ ]:
# test on validation data
y_est = pipeline.predict(val)
plt.scatter(val["VOLUME"], y_est)

In [ ]:
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(val["VOLUME"], y_est))